In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/KeepCoding/NLP/Práctica

/content/drive/MyDrive/KeepCoding/NLP/Práctica


In [4]:
import pandas as pd
import random
import re
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
from nltk.tokenize import TreebankWordTokenizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, precision_recall_curve # Reporte
from sklearn.model_selection import train_test_split # Modelado
from sklearn.pipeline import Pipeline # Modelado
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer # Modelado
from sklearn.linear_model import LogisticRegression # Reporte



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [5]:
df_reviews_filtered = pd.read_csv("df_reviews_filtered.csv")

In [6]:
# Stopwords estándar
custom_stopwords = set(stopwords.words("english"))

# Conservamos negaciones importantes para análisis de sentimiento
custom_stopwords.discard("no")
custom_stopwords.discard("not")
custom_stopwords.discard("nor")

# Añadimos stopwords detectadas durante la exploración
custom_stopwords.update(["use", "used", "one"])

In [7]:
def review_to_words(review):

    # Convertimos a minúscula y quitamos todo lo que no sea texto o números
    text = re.sub(r"[^a-zA-Z0-9]", " ", review.lower())
    # Dividimos en tokens por espacios
    tokenizer = TreebankWordTokenizer()
    words = tokenizer.tokenize(text)
    # Eliminamos stopwords
    words = [w for w in words if w not in custom_stopwords]

    return " ".join(words)

In [8]:
df_reviews_filtered.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,sentiment_label
0,A3MCKE2J5F9XO9,B000R845C4,ME,"[0, 0]",Stripper clips. Glorious stripper clips. The...,3.0,Stripper clips...,1374019200,"07 17, 2013",0
1,A7OHBCMRMNVPR,B001B8ONZI,Paul B.,"[0, 0]",Purchases two pair of these. Both pair worked ...,3.0,50%,1332115200,"03 19, 2012",0
2,AEG9SBQ1UETBD,B0015TT3V2,R. Wasilewski,"[0, 1]",I put this on my fox float 32 and honestly not...,3.0,Not sure it really works,1380067200,"09 25, 2013",0
3,A3A2XWUX499M57,B000DZD3FQ,Mikhael,"[0, 0]",It's a good computer. My rating is impacted by...,2.0,Programming is a pain. Easy to accidentally cl...,1356912000,"12 31, 2012",0
4,A43Y8KFO5JK5P,B002HU086C,"Ahmed Syed ""Genus: Constrictus realtight and ...","[1, 3]",These are heavy duty stakes which certainly ar...,1.0,Good but cheap plastic,1369699200,"05 28, 2013",0


In [9]:
processed_reviews = df_reviews_filtered['reviewText'].apply(review_to_words)

In [10]:
print('Review original: {}'.format(df_reviews_filtered['reviewText'].values[1]))
print('Review procesada: {}'.format(processed_reviews[1]))

Review original: Purchases two pair of these. Both pair worked perfectly at first, then one pair quit completely. Changed the batteries with no help. FUBAR. Sent back for an exchange. Hope it was just an odd occurance and the replacment will be great. Amazon was great as always.
Review procesada: purchases two pair pair worked perfectly first pair quit completely changed batteries no help fubar sent back exchange hope odd occurance replacment great amazon great always


In [11]:
df_reviews_filtered.loc[:, 'processedReview'] = processed_reviews

In [12]:
df_reviews_filtered.head(5)

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,sentiment_label,processedReview
0,A3MCKE2J5F9XO9,B000R845C4,ME,"[0, 0]",Stripper clips. Glorious stripper clips. The...,3.0,Stripper clips...,1374019200,"07 17, 2013",0,stripper clips glorious stripper clips chew fi...
1,A7OHBCMRMNVPR,B001B8ONZI,Paul B.,"[0, 0]",Purchases two pair of these. Both pair worked ...,3.0,50%,1332115200,"03 19, 2012",0,purchases two pair pair worked perfectly first...
2,AEG9SBQ1UETBD,B0015TT3V2,R. Wasilewski,"[0, 1]",I put this on my fox float 32 and honestly not...,3.0,Not sure it really works,1380067200,"09 25, 2013",0,put fox float 32 honestly noticed no differenc...
3,A3A2XWUX499M57,B000DZD3FQ,Mikhael,"[0, 0]",It's a good computer. My rating is impacted by...,2.0,Programming is a pain. Easy to accidentally cl...,1356912000,"12 31, 2012",0,good computer rating impacted user interface f...
4,A43Y8KFO5JK5P,B002HU086C,"Ahmed Syed ""Genus: Constrictus realtight and ...","[1, 3]",These are heavy duty stakes which certainly ar...,1.0,Good but cheap plastic,1369699200,"05 28, 2013",0,heavy duty stakes certainly big upgrade stakes...


# Modelado

## Separamos en conjunto de train y test

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    df_reviews_filtered['processedReview'],
    df_reviews_filtered['sentiment_label'],
    train_size=0.75,
    test_size=0.25,
    random_state=42,
    shuffle=True
)

In [14]:
X_train.iloc[:5]

,processedReview
4901,knife good sharpen arrived keeping edge well n...
4375,dont like way fits
6698,great sleeping bag not much room move feet leg...
9805,old murray falcon brand 5 speed freewheel need...
1101,portable water bucket made decent quality mate...


In [15]:
y_train.iloc[:5]

,sentiment_label
4901,0
4375,0
6698,1
9805,1
1101,0


In [16]:
cv = TfidfVectorizer(
    max_df=0.95,
    min_df=3,
    max_features=2500,
    strip_accents='ascii',
    ngram_range=(1, 1)
)
cv.fit(X_train)

TfidfVectorizer(max_df=0.95, max_features=2500, min_df=3, strip_accents='ascii')

#### max_df=0.95 elimina palabras excesivamente frecuentes presentes en la mayoría de documentos.

#### min_df=3 evita palabras demasiado raras que podrían introducir ruido.

#### max_features=2500 limita la dimensionalidad del problema.

#### ngram_range=(1,1) se ha seleccionado para trabajar únicamente con unigramas y mantener un modelo sencillo.

## TF-IDF scores del training set y test set

In [17]:
X_train_ = cv.transform(X_train)
X_test_ = cv.transform(X_test)

In [18]:
i = random.randint(0, len(X_train))
doc_vector = X_train_[i]
df_tfidf = pd.DataFrame(doc_vector.T.todense(), index=cv.get_feature_names_out(), columns=['tfidf'])
df_tfidf = df_tfidf[df_tfidf['tfidf'] > 0]

top_n = 10
print('Top {} words with highest TF_IDF in the review {}:\n{}'.format(top_n, i, df_tfidf.sort_values(by=["tfidf"],ascending=False)[:top_n]))
print('\nTop {} words with lowest TF_IDF in the review {}:\n{}'.format(top_n, i, df_tfidf.sort_values(by=["tfidf"],ascending=False)[-top_n:]))

Top 10 words with highest TF_IDF in the review 3241:
                   tfidf
seen            0.324514
premium         0.225664
disappointment  0.222790
seams           0.216539
moisture        0.214336
disappointing   0.207611
threads         0.194803
glove           0.187907
considering     0.186046
coming          0.179873

Top 10 words with lowest TF_IDF in the review 3241:
              tfidf
hands      0.150830
seem       0.146458
reviews    0.143459
purchased  0.134081
might      0.132167
however    0.123020
hard       0.121704
light      0.113771
price      0.102167
good       0.079721


# Entrenamiento

In [19]:
c_params = [0.01, 0.05, 0.25, 0.5, 1, 10, 100, 1000, 10000]

train_acc = list()
test_acc = list()
for c in c_params:
    lr = LogisticRegression(C=c, solver='lbfgs', max_iter=500)
    lr.fit(X_train_, y_train)

    train_predict = lr.predict(X_train_)
    test_predict = lr.predict(X_test_)

    print ("Accuracy for C={}: {}".format(c, accuracy_score(y_test, test_predict)))

    train_acc.append(accuracy_score(y_train, train_predict))
    test_acc.append(accuracy_score(y_test, test_predict))

Accuracy for C=0.01: 0.7472
Accuracy for C=0.05: 0.7628
Accuracy for C=0.25: 0.7784
Accuracy for C=0.5: 0.78
Accuracy for C=1: 0.7856
Accuracy for C=10: 0.7648
Accuracy for C=100: 0.7368
Accuracy for C=1000: 0.718
Accuracy for C=10000: 0.71


#### Los resultados obtenidos muestran que el parámetro de regularización `C`
#### influye significativamente en el rendimiento del modelo.

#### Valores muy pequeños de `C` generan modelos excesivamente restringidos,
#### mientras que valores demasiado grandes provocan una pérdida de capacidad
#### de generalización y posibles problemas de overfitting.

#### El mejor resultado se obtiene aproximadamente con `C = 1`, alcanzando una
#### accuracy cercana al 78.5%, lo que representa un equilibrio razonable entre
#### simplicidad del modelo y capacidad predictiva.

In [20]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

alpha_params = [0.01, 0.1, 0.5, 1, 5, 10]

train_acc_nb = list()
test_acc_nb = list()

for alpha in alpha_params:

    nb = MultinomialNB(alpha=alpha)

    nb.fit(X_train_, y_train)

    train_predict = nb.predict(X_train_)
    test_predict = nb.predict(X_test_)

    print("Accuracy for alpha={}: {}".format(
        alpha,
        accuracy_score(y_test, test_predict)
    ))

    train_acc_nb.append(
        accuracy_score(y_train, train_predict)
    )

    test_acc_nb.append(
        accuracy_score(y_test, test_predict)
    )

Accuracy for alpha=0.01: 0.7608
Accuracy for alpha=0.1: 0.7628
Accuracy for alpha=0.5: 0.764
Accuracy for alpha=1: 0.7672
Accuracy for alpha=5: 0.77
Accuracy for alpha=10: 0.77


#### Se ha entrenado también un modelo Multinomial Naive Bayes utilizando distintos valores del parámetro `alpha`, encargado de aplicar suavizado(smoothing) para evitar problemas con palabras poco frecuentes o no observadas durante el entrenamiento.

#### Los resultados muestran una mejora progresiva del rendimiento a medida que aumenta el valor de `alpha`, estabilizándose alrededor de los valores 5 y 10 con una accuracy cercana al 77%.

#### Comparando ambos modelos, Logistic Regression obtiene un mejor resultado (78.56%) frente a Multinomial Naive Bayes (77.00%), por lo que será el modelo seleccionado para las siguientes etapas del proyecto.

#### Aunque la diferencia no es muy elevada, Logistic Regression parece ser capaz de capturar mejor las relaciones presentes en las representaciones ,TF-IDF generadas a partir de las reviews.

# 4. Reporte de métricas y conclusiones

#### Tras comparar los modelos entrenados durante la etapa anterior, se ha seleccionado Logistic Regression como modelo final al obtener el mejor rendimiento sobre el conjunto de test.

#### Tras evaluar distintos valores del parámetro C, se selecciona C=1 al ser el valor que obtiene la mayor accuracy sobre el conjunto de test (78.56%). A continuación se evalúa el modelo seleccionado mediante las métricas finales de clasificación.

In [23]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

# Modelo seleccionado tras la búsqueda de hiperparámetros
lr_best = LogisticRegression(
    C=1,
    solver='lbfgs',
    max_iter=500
)

lr_best.fit(X_train_, y_train)

# Predicciones sobre el conjunto de test
y_pred = lr_best.predict(X_test_)

# Métricas
print("=== MATRIZ DE CONFUSIÓN ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

print("\n=== ACCURACY ===")
print(f"{accuracy_score(y_test, y_pred):.4f}")

=== MATRIZ DE CONFUSIÓN ===
[[1018  254]
 [ 282  946]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       0.78      0.80      0.79      1272
           1       0.79      0.77      0.78      1228

    accuracy                           0.79      2500
   macro avg       0.79      0.79      0.79      2500
weighted avg       0.79      0.79      0.79      2500


=== ACCURACY ===
0.7856


#### La matriz de confusión muestra que el modelo es capaz de clasificar correctamente la mayoría de las reviews de ambas clases. Se observan 1018 reviews negativas y 946 positivas clasificadas correctamente, mientras que los errores se reparten de forma relativamente equilibrada entre falsos positivos y falsos negativos.

#### En cuanto a las métricas de clasificación, tanto la precisión como el recall presentan valores cercanos al 80% para ambas clases, lo que indica que el modelo mantiene un comportamiento equilibrado sin favorecer de forma significativa ninguna de ellas.

#### El modelo alcanza una accuracy de 78.56%, obteniendo además valores de F1-score próximos a 0.79 para ambas clases. Estos resultados sugieren una capacidad razonable para identificar el sentimiento de las reviews utilizando una representación TF-IDF y técnicas clásicas de Machine Learning.

#### Considerando el objetivo de la práctica y la simplicidad del enfoque empleado, los resultados obtenidos pueden considerarse satisfactorios.

# Conclusión

#### En conclusión, el objetivo de construir un clasificador binario de sentimiento sobre reviews de Amazon se ha cumplido satisfactoriamente.

#### La exploración inicial del corpus, junto con las decisiones tomadas durante el preprocesado y la vectorización del texto, han permitido obtener un modelo capaz de identificar el sentimiento de las reviews con una precisión razonable.

#### Como posibles líneas de mejora futuras podrían explorarse técnicas más avanzadas basadas en embeddings o modelos Transformer, así como procesos adicionales de optimización de hiperparámetros.